In [1]:
import os
from pathlib import Path

ROOT = Path("/kaggle/working/bac_assistant")
(ROOT / "data").mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

Path("requirements.txt").write_text(
    "google-genai\ngradio\npython-dotenv\n"
)
!pip install -q -r requirements.txt

print("Working in:", os.getcwd())

Working in: /kaggle/working/bac_assistant


In [2]:
%%writefile config.py
"""Central configuration. Works the same locally, on Colab, Kaggle or HF Spaces."""
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

# One sub-folder per exam year: data/2022, data/2023, ...
DATA_DIR = Path(os.getenv("BAC_DATA_DIR", BASE_DIR / "data"))
CACHE_DIR = Path(os.getenv("BAC_CACHE_DIR", BASE_DIR / ".cache"))

# Verify exact names available to your key with client.models.list()
ROUTER_MODEL = os.getenv("ROUTER_MODEL", "gemini-3.5-flash")
GENERATOR_MODEL = os.getenv("GENERATOR_MODEL", "gemini-3.8-flash")


def get_api_key() -> str:
    """Env var first, then Kaggle / Colab secrets as a fallback."""
    key = os.getenv("GEMINI_API_KEY")
    if key:
        return key
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GEMINI_API_KEY")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    raise RuntimeError("GEMINI_API_KEY not found. Set it as an env variable or in .env")

Writing config.py


In [3]:
import shutil, re
from pathlib import Path

INPUT = Path("/kaggle/input")
shutil.rmtree("data", ignore_errors=True)   # امسح التوزيع الخاطئ

year_re = re.compile(r"^((?:19|20)\d\d)_")

# 1) ملفات JSON
for f in INPUT.rglob("*.json"):
    m = year_re.match(f.name)
    if m and f.name.endswith(("sujet.json", "correction.json", "figures_index.json")):
        d = Path("data") / m.group(1)
        d.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, d / f.name)

# 2) الصور (حسب السنة في بداية الاسم)
for f in INPUT.rglob("*"):
    if f.suffix.lower() in {".png", ".jpg", ".jpeg"}:
        m = year_re.match(f.name)
        if m:
            d = Path("data") / m.group(1) / "images"
            d.mkdir(parents=True, exist_ok=True)
            shutil.copy(f, d / f.name)

for y in sorted(Path("data").iterdir()):
    n_img = len(list((y / "images").glob("*"))) if (y / "images").exists() else 0
    print(y.name, "->", len(list(y.glob("*.json"))), "json |", n_img, "images")

2019 -> 3 json | 3 images
2021 -> 3 json | 4 images
2022 -> 3 json | 8 images


In [4]:
import json
from pathlib import Path

for y in sorted(Path("data").iterdir()):
    fp = next(y.glob("*figures_index.json"))
    fi = json.load(open(fp, encoding="utf-8"))
    fi = fi if isinstance(fi, list) else next(v for v in fi.values() if isinstance(v, list))
    missing = [Path(x["file"]).name for x in fi if not (y / "images" / Path(x["file"]).name).exists()]
    print(f"{y.name}: {len(fi)} figures in index, {len(missing)} missing")
    for m in missing[:5]:
        print("   ✗", m)

print("\nمثال من الـ index:", fi[0])

2019: 3 figures in index, 0 missing
2021: 4 figures in index, 0 missing
2022: 8 figures in index, 0 missing

مثال من الـ index: {'id': '2022_sujet_s1_ex1_cf', 'file': 'figures/2022_sujet_s1_ex1_cf.png', 'year': 2022, 'document': 'sujet', 'source_pdf': '2022_sujet.pdf', 'subject': 1, 'exercise': 1, 'exercise_id': '2022_s1_ex1', 'page_in_source_pdf': 1, 'page_in_original_big_pdf': 5, 'role': 'given', 'description_ar': 'المنحنى (C_f) والمماس (T) والمستقيم المقارب (Δ) للتمرين الأول', 'description_fr': "Courbe (Cf), tangente (T) et asymptote (Δ) de l'exercice 1 (S1)", 'width': 338, 'height': 269}


In [5]:
from kaggle_secrets import UserSecretsClient
try:
    print("OK, key length:", len(UserSecretsClient().get_secret("GEMINI_API_KEY")))
except Exception as e:
    print("ERROR:", e)

OK, key length: 53


In [ ]:
c

In [6]:
import importlib, json, collections
from pathlib import Path
import config, loader
importlib.reload(config); importlib.reload(loader)

res = loader.load_all(use_cache=False)
print("Years    :", res.years)
print("Exercises:", len(res.chunks))
for c in res.catalog:
    print(f'  {c["id"]:<20} S{c["subject"]} Ex{c["exercise_number"]} | {c["points"]} pts | {c["topic"][:45]}')
print("Warnings :", len(res.warnings))
for w in res.warnings:
    print("  !", w)

# قيم حقل document في الـ index (نتأكد أن صور التصحيح تتصنف صح)
vals = collections.Counter()
for fp in Path("data").rglob("*figures_index.json"):
    d = json.load(open(fp, encoding="utf-8"))
    d = d if isinstance(d, list) else next(v for v in d.values() if isinstance(v, list))
    vals.update(x.get("document") for x in d)
print("document values:", dict(vals))

# عدد الصور المرتبطة بكل نوع
print("figures sujet:", sum(len(c["figures_sujet"]) for c in res.chunks),
      "| figures corr:", sum(len(c["figures_corr"]) for c in res.chunks))

ModuleNotFoundError: No module named 'loader'

In [ ]:
%%writefile router.py
"""Router: first Gemini call. Understands the student's message and decides what to do.

Gemini only *decides* (intent + which exercise ids). Python then validates the decision,
so a hallucinated id or a wrong year can never reach the generator.
"""
import enum
import json
from dataclasses import dataclass, field
from typing import List, Optional

from pydantic import BaseModel, Field

import config


# ---------- what Gemini must return ----------

class Intent(str, enum.Enum):
    solve_exercise = "solve_exercise"            # full exercise from the dataset
    solve_subquestion = "solve_subquestion"      # one sub-question of a dataset exercise
    suggest_exercises = "suggest_exercises"      # "give me exercises about complex numbers"
    explain_concept = "explain_concept"          # explain a lesson (optionally using a dataset exercise)
    general = "general"                          # anything else: normal math/bac chat, or not in dataset
    needs_clarification = "needs_clarification"  # really cannot proceed without asking


class RouterDecision(BaseModel):
    intent: Intent
    language: str = Field(description="Language of the student's message: ar, darija, fr, en, or mixed")
    standalone_question: str = Field(
        description="The student's request rewritten so it is fully self-contained "
                    "(resolve 'the second question', 'the same exercise' using the history). "
                    "Same language as the student.")
    year: Optional[int] = Field(default=None, description="Exam year requested, if any")
    subject: Optional[int] = Field(default=None, description="1 or 2 (first/second subject), if given")
    exercise_number: Optional[int] = Field(default=None, description="Exercise number, if given")
    sub_question: Optional[str] = Field(default=None, description="Sub-question label such as '2' or 'أ', if given")
    selected_ids: List[str] = Field(
        default_factory=list,
        description="Ids copied EXACTLY from the catalog. Empty if nothing in the catalog matches.")
    clarification: Optional[str] = Field(
        default=None, description="Only for needs_clarification: one short question to the student, in their language")


SYSTEM_TEMPLATE = """You are the routing brain of an Algerian Baccalaureate (BAC) mathematics assistant.
You do NOT answer the student. You only decide what to do, and return JSON.

The student may write Modern Standard Arabic, Algerian Darija, French, English, or a mix,
with typos. Understand forms like: "التمرين التالت", "exo 3", "الموضوع الثاني", "sujet 2", "s2", "باك 2022".
"subject" means الموضوع الأول (1) or الموضوع الثاني (2).

DATASET CATALOG (years available: {years}). Format: id | year | subject | exercise | points | topic | tags
{catalog}

INTENT RULES
- solve_exercise: the student wants a specific catalog exercise solved (or says it is the one they mean). selected_ids = exactly 1 id.
- solve_subquestion: same, but only one sub-question. selected_ids = 1 id, and fill sub_question.
- suggest_exercises: the student wants practice exercises on a topic. selected_ids = up to 5 ids, best match first.
- explain_concept: the student wants a lesson/concept explained. selected_ids = 0-2 catalog ids ONLY if one is a really good example, else empty.
- general: everything else: simple math questions, other exercises not in the catalog, a year that is not in the catalog,
  exercises the student pasted in the message, study advice, greetings. selected_ids = [].
- needs_clarification: use ONLY when you truly cannot proceed, for example the exercise number is given but the
  year or subject is missing and both could exist. Prefer a reasonable guess over asking.

IMPORTANT
- NEVER invent an id. Copy ids exactly from the catalog or leave selected_ids empty.
- If the student asks for a year that is not in the catalog, use intent=general and keep year = that year.
- Use the conversation history and the "last exercise discussed" to resolve follow-ups such as "والسؤال 2؟" or "explain more".
- standalone_question must contain everything needed to answer without the history.
"""


# ---------- result after Python validation ----------

@dataclass
class Route:
    intent: str
    standalone_question: str
    language: str = "unknown"
    ids: list = field(default_factory=list)       # validated chunk ids
    use_dataset: bool = False
    year: Optional[int] = None
    sub_question: Optional[str] = None
    clarification: Optional[str] = None
    notes: list = field(default_factory=list)     # why Python changed/ignored something (debug)


# ---------- helpers ----------

def format_catalog(catalog) -> str:
    lines = []
    for c in catalog:
        tags = ", ".join(c.get("topic_tags") or [])
        lines.append(f'{c["id"]} | {c["year"]} | S{c["subject"]} | Ex{c["exercise_number"]} | '
                     f'{c["points"]}pts | {c["topic"]} | {tags}')
    return "\n".join(lines)


def format_history(history, max_turns=4, max_chars=300) -> str:
    if not history:
        return "(none)"
    out = []
    for m in history[-max_turns:]:
        role = "Student" if m.get("role") == "user" else "Assistant"
        out.append(f'{role}: {str(m.get("content", ""))[:max_chars]}')
    return "\n".join(out)


def _call_router(client, system: str, contents: str) -> str:
    """One Gemini call that returns the raw JSON text. Isolated so it is easy to mock/test."""
    from google.genai import types
    resp = client.models.generate_content(
        model=config.ROUTER_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system,
            response_mime_type="application/json",
            response_schema=RouterDecision,
            temperature=0.0,
        ),
    )
    return resp.text


def _validate(decision: RouterDecision, by_id: dict) -> Route:
    notes = []
    intent = decision.intent.value

    # keep only ids that really exist (dedupe, keep order)
    ids = []
    for i in decision.selected_ids:
        if i in by_id and i not in ids:
            ids.append(i)
        elif i not in by_id:
            notes.append(f"dropped unknown id: {i}")

    # if a year was requested, ids from another year are a hallucination for solve_* intents
    if intent in ("solve_exercise", "solve_subquestion") and decision.year:
        good = [i for i in ids if by_id[i]["year"] == decision.year]
        if len(good) != len(ids):
            notes.append("dropped ids not matching the requested year")
        ids = good

    if intent in ("solve_exercise", "solve_subquestion"):
        ids = ids[:1]
        if not ids:
            intent = "general"
            notes.append("exercise not found in dataset -> answer from model knowledge")
    elif intent == "suggest_exercises":
        ids = ids[:5]
        if not ids:
            intent = "general"
            notes.append("no matching exercises -> answer from model knowledge")
    elif intent == "explain_concept":
        ids = ids[:2]
    elif intent == "general":
        ids = []

    clarification = decision.clarification
    if intent == "needs_clarification" and not clarification:
        clarification = "Peux-tu préciser l'année et le sujet (1 ou 2) ? / وضّحلي السنة والموضوع (1 ولا 2)؟"

    return Route(
        intent=intent,
        standalone_question=decision.standalone_question,
        language=decision.language,
        ids=ids,
        use_dataset=bool(ids),
        year=decision.year,
        sub_question=decision.sub_question,
        clarification=clarification,
        notes=notes,
    )


# ---------- public API ----------

def route(question: str, history, loaded, client, last_ids=None) -> Route:
    """loaded = loader.LoadResult. Never raises: on any failure it falls back to intent=general."""
    system = SYSTEM_TEMPLATE.format(
        years=", ".join(map(str, loaded.years)),
        catalog=format_catalog(loaded.catalog),
    )
    contents = (
        f"Conversation so far:\n{format_history(history)}\n\n"
        f"Last exercise(s) discussed: {', '.join(last_ids) if last_ids else '(none)'}\n\n"
        f"New student message:\n{question}"
    )
    by_id = loaded.by_id()

    last_err = None
    for _ in range(2):  # one retry
        try:
            raw = _call_router(client, system, contents)
            decision = RouterDecision.model_validate_json(raw)
            return _validate(decision, by_id)
        except Exception as e:  # network, quota, bad JSON...
            last_err = e

    return Route(intent="general", standalone_question=question,
                 notes=[f"router failed, fallback to general: {last_err!r}"])

In [ ]:
import importlib, config, loader, router
from google import genai
importlib.reload(config); importlib.reload(loader); importlib.reload(router)

client = genai.Client(api_key=config.get_api_key())
loaded = loader.load_all()

tests = [
    "حلي لي التمرين التالت تاع باك 2022 الموضوع الأول",
    "exo 4 sujet 2 2019",
    "عطيني تمارين على الأعداد المركبة",
    "شرحلي كيفاش نحسب نهاية دالة لوغاريتمية",
    "حلي لي تمرين الاحتمالات تاع 2023",                 # سنة ما كايناش
    "واش هو مشتق x^2 * e^x ؟",                         # سؤال عادي
    "salut, comment tu vas ?",
    "حلي لي التمرين 2",                                  # ناقص السنة والموضوع
]
for q in tests:
    r = router.route(q, history=[], loaded=loaded, client=client)
    print(f"Q: {q}\n   -> {r.intent} | ids={r.ids} | year={r.year} | lang={r.language}")
    if r.clarification: print("   ?", r.clarification)
    if r.notes: print("   notes:", r.notes)
    print()

# اختبار المتابعة (follow-up)
hist = [{"role": "user", "content": "حلي لي التمرين 3 تاع 2022 موضوع 1"},
        {"role": "assistant", "content": "...حل التمرين..."}]
r = router.route("والسؤال الثاني كيفاش؟", hist, loaded, client, last_ids=["2022_s1_ex3"])
print("Follow-up ->", r.intent, r.ids, "| sub_question =", r.sub_question)
print("standalone:", r.standalone_question)

In [ ]:
c

In [ ]:
import time, importlib, router
importlib.reload(router)

tests = [
    ("واش هو مشتق x^2 * e^x ؟", [], None),
    ("حلي لي التمرين 2", [], None),
    ("والسؤال الثاني كيفاش؟",
     [{"role": "user", "content": "حلي لي التمرين 3 تاع 2022 موضوع 1"},
      {"role": "assistant", "content": "...حل التمرين..."}],
     ["2022_s1_ex3"]),
]
for q, hist, last in tests:
    r = router.route(q, hist, loaded, client, last_ids=last)
    print(f"Q: {q}\n   -> {r.intent} | ids={r.ids} | sub={r.sub_question} | lang={r.language}")
    print("   standalone:", r.standalone_question)
    if r.clarification: print("   ?", r.clarification)
    if r.notes: print("   notes:", r.notes)
    print()
    time.sleep(13)

In [ ]:
%%writefile generator.py
"""Generator: second Gemini call. Writes the answer for the student.

Not every route needs Gemini:
  - needs_clarification / error -> the router's message is returned as is
  - suggest_exercises           -> built by Python from the catalog
  - solve_* / explain / general -> Gemini (with the official correction and the figures when available)
"""
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import config
from router import (MAX_ATTEMPTS, MAX_WAIT_SECONDS, _is_quota_error, _retry_delay,
                    format_history)

MODES = ("full", "hints")
MAX_IMAGES = 6


@dataclass
class Answer:
    text: str
    ids: list = field(default_factory=list)       # dataset exercises used
    figures: list = field(default_factory=list)   # image paths of the statement, so the UI can show them
    used_gemini: bool = False
    notes: list = field(default_factory=list)


# ---------- prompts ----------

BASE_RULES = """You are a friendly and rigorous tutor for the Algerian Baccalaureate (BAC) in mathematics.

LANGUAGE
- Answer in the language of the student's message. If the student writes Arabic or Algerian Darija,
  answer in clear Modern Standard Arabic with simple explanations, like a good teacher. If French, answer in French.
- Write every formula in LaTeX between dollar signs, e.g. $f'(x)=e^x(x+2)$.

QUALITY
- Be accurate. Double-check calculations. Show the key steps a BAC student must write.
- Never invent the text of an exam exercise. If you are not sure of an exam statement, say so and ask the student to paste it.
"""

DATASET_RULES = """
DATASET EXERCISE
- The OFFICIAL CORRECTION in the context is the reference. Your final results must match it.
  Explain it as a clear worked solution (you may add reasoning between its steps), but never change its results.
  If it seems incomplete or inconsistent, say so briefly instead of silently changing it.
- Use ONLY the statement, descriptions and images provided. Read values from the images carefully;
  if something is not readable, say so.
"""

MODE_FULL = """
MODE: FULL SOLUTION
- Give a complete, well organized solution. Label every sub-question and highlight each final result.
"""

MODE_HINTS = """
MODE: HINTS ONLY
- Do NOT give the full solution and do NOT reveal final results.
- Give: (1) the lesson or theorem to use, (2) the first step or a guiding question, (3) one short hint.
- End by telling the student they can switch to the "full solution" option to see the whole answer.
"""

NOT_IN_DATASET = """
NOTE: The student asks about an exam ({year}) that is NOT in your database. Start by saying so briefly,
then help from your own knowledge. If you do not remember the exact statement, ask the student to paste it.
"""


# ---------- context ----------

def _figures_text(figs, label):
    lines = [f"- {f.get('description_ar') or f.get('description_fr') or '(figure)'}" for f in figs]
    return f"{label}:\n" + "\n".join(lines) + "\n" if lines else ""


def build_exercise_context(chunk) -> str:
    return (
        f'=== EXERCISE: BAC {chunk["year"]}, subject {chunk["subject"]}, exercise {chunk["exercise_number"]} '
        f'({chunk["points"]} pts) ===\n'
        f'Topic: {chunk["topic"]}\n\nSTATEMENT:\n{chunk["statement"]}\n\n'
        f'{_figures_text(chunk["figures_sujet"], "Figures in the statement (descriptions)")}'
        f'\n=== OFFICIAL CORRECTION ===\n{chunk["solution"] or "(no correction available: solve it yourself)"}\n\n'
        f'{_figures_text(chunk["figures_corr"], "Figures in the correction (descriptions)")}'
    )


def _image_parts(chunk, mode, notes):
    """Statement figures always; correction figures only in full mode (so hints don't leak the answer)."""
    figs = list(chunk["figures_sujet"]) + (list(chunk["figures_corr"]) if mode == "full" else [])
    parts = []
    for f in figs[:MAX_IMAGES]:
        p = Path(f["path"])
        if not p.exists():
            notes.append(f"image not found: {p.name}")
            continue
        mime = "image/jpeg" if p.suffix.lower() in (".jpg", ".jpeg") else "image/png"
        parts.append({"mime": mime, "data": p.read_bytes()})
    return parts


def _suggest_text(route, by_id) -> str:
    lines = ["هذه تمارين من الباك تناسب طلبك:", ""]
    for i, cid in enumerate(route.ids, 1):
        c = by_id[cid]
        lines.append(f'{i}. **باك {c["year"]} – الموضوع {c["subject"]} – التمرين {c["exercise_number"]}** '
                     f'({c["points"]} نقاط): {c["topic"]}')
    lines += ["", "قولي أي تمرين تحب نحلوه معاك، مثلا: «حلي لي التمرين 3 تاع 2019 الموضوع الأول»."]
    return "\n".join(lines)


# ---------- Gemini call ----------

def _call_gemini(client, system: str, parts: list) -> str:
    """parts: list of str or {'mime','data'} dicts. Isolated so it can be mocked in tests."""
    from google.genai import types
    contents = []
    for p in parts:
        if isinstance(p, str):
            contents.append(types.Part.from_text(text=p))
        else:
            contents.append(types.Part.from_bytes(data=p["data"], mime_type=p["mime"]))
    resp = client.models.generate_content(
        model=config.GENERATOR_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(system_instruction=system, temperature=0.3),
    )
    return resp.text


# ---------- public API ----------

def generate(route, loaded, client, mode: str = "full", history=None) -> Answer:
    """Never raises. mode: 'full' (complete solution) or 'hints'."""
    if mode not in MODES:
        mode = "full"
    by_id = loaded.by_id()

    # 1) no Gemini needed
    if route.intent in ("needs_clarification", "error"):
        return Answer(text=route.clarification or "")
    if route.intent == "suggest_exercises":
        return Answer(text=_suggest_text(route, by_id), ids=route.ids)

    # 2) build the Gemini request
    notes, images, figures = [], [], []
    system = BASE_RULES
    context = ""

    chunks = [by_id[i] for i in route.ids if i in by_id]
    if route.intent in ("solve_exercise", "solve_subquestion") and chunks:
        system += DATASET_RULES + (MODE_HINTS if mode == "hints" else MODE_FULL)
        context = build_exercise_context(chunks[0])
        images = _image_parts(chunks[0], mode, notes)
        figures = [f["path"] for f in chunks[0]["figures_sujet"] if Path(f["path"]).exists()]
        if route.intent == "solve_subquestion" and route.sub_question:
            context += f'\nThe student asks ONLY about sub-question "{route.sub_question}". Answer only that part ' \
                       f'(recall earlier results briefly if needed).\n'
    elif route.intent == "explain_concept" and chunks:
        context = "Example exercise you may use to illustrate (optional):\n" + build_exercise_context(chunks[0])
        images = _image_parts(chunks[0], "hints", notes)
        figures = []
    else:  # general (or anything else)
        if route.year and route.year not in loaded.years:
            system += NOT_IN_DATASET.format(year=route.year)
        if mode == "hints":
            system += "\nIf the student asks to solve a problem, prefer guiding hints over the full solution.\n"

    text = (
        f"Conversation so far:\n{format_history(history, max_turns=6, max_chars=500)}\n\n"
        f"{context}\nStudent's request:\n{route.standalone_question}"
    )
    parts = [text] + images

    # 3) call, with patience for quota errors
    last_err = None
    for attempt in range(MAX_ATTEMPTS):
        try:
            out = _call_gemini(client, system, parts)
            return Answer(text=out, ids=route.ids if chunks else [], figures=figures,
                          used_gemini=True, notes=notes)
        except Exception as e:
            last_err = e
            if _is_quota_error(e):
                wait = _retry_delay(e) + 1
                if wait > MAX_WAIT_SECONDS or attempt == MAX_ATTEMPTS - 1:
                    break
                time.sleep(wait)
            else:
                break
    msg = ("الخدمة مشغولة حاليا (حد عدد الطلبات). عاود المحاولة بعد دقيقة."
           if _is_quota_error(last_err) else "صرات مشكلة تقنية. عاود المحاولة من فضلك.")
    return Answer(text=msg, notes=notes + [f"generation failed: {last_err!r}"[:300]])

In [ ]:
import time, importlib, router, generator
importlib.reload(router); importlib.reload(generator)
from IPython.display import Markdown, display

def ask(q, mode="full", history=None, last_ids=None):
    r = router.route(q, history or [], loaded, client, last_ids=last_ids)
    a = generator.generate(r, loaded, client, mode=mode, history=history)
    print(f"[{r.intent} | ids={r.ids} | gemini={a.used_gemini} | mode={mode} | notes={r.notes + a.notes}]")
    display(Markdown(a.text))
    return r, a

# 1) تمرين فيه منحنى (2022 s1 ex1) بوضع التلميحات
ask("حلي لي التمرين الأول تاع باك 2022 الموضوع الأول", mode="hints")
time.sleep(30)

# 2) نفس التمرين بالحل الكامل
ask("حلي لي التمرين الأول تاع باك 2022 الموضوع الأول", mode="full")
time.sleep(30)

# 3) اقتراح تمارين (هذا بدون استدعاء Gemini للـ generator)
ask("عطيني تمارين على الأعداد المركبة")

In [ ]:
%%writefile llm_utils.py
"""Shared helpers for calling Gemini safely: retries, back-off and fallback models."""
import random
import re
import time

MAX_WAIT_SECONDS = 30   # a longer wait (e.g. daily quota) is not worth blocking a student for
TRANSIENT_CODES = {500, 502, 503, 504}


def is_quota_error(err) -> bool:
    """429: too many requests / quota exceeded."""
    return getattr(err, "code", None) == 429 or "RESOURCE_EXHAUSTED" in str(err)


def is_transient_error(err) -> bool:
    """Temporary server-side problem (overloaded, unavailable, timeout...). Worth retrying."""
    if getattr(err, "code", None) in TRANSIENT_CODES:
        return True
    s = str(err)
    return any(k in s for k in ("UNAVAILABLE", "DEADLINE_EXCEEDED", "high demand", "overloaded"))


def retry_delay(err, default: float = 10.0) -> float:
    """Seconds Gemini asks us to wait, parsed from the error text."""
    s = str(err)
    m = re.search(r"retry in ([\d.]+)\s*(ms|s)\b", s, re.I)
    if m:
        v = float(m.group(1))
        return v / 1000 if m.group(2).lower() == "ms" else v
    m = re.search(r"retryDelay[^\d]*([\d.]+)s", s)
    return float(m.group(1)) if m else default


def call_with_retry(call, models, attempts_per_model: int = 3):
    """call(model) -> result. Tries each model in order.
      - quota (429)       : waits what Gemini asks; if too long, moves to the next model
      - transient (503...) : back-off 2s, 4s..., then moves to the next model
      - any other error    : stops immediately (retrying would not help)
    Returns (result, None, model_used) on success, or (None, last_error, None)."""
    last_err = None
    for model in models:
        for attempt in range(attempts_per_model):
            try:
                return call(model), None, model
            except Exception as e:
                last_err = e
                last = attempt == attempts_per_model - 1
                if is_quota_error(e):
                    wait = retry_delay(e) + 1
                    if wait > MAX_WAIT_SECONDS or last:
                        break
                    time.sleep(wait)
                elif is_transient_error(e):
                    if last:
                        break
                    time.sleep(2 * 2 ** attempt + random.random())
                else:
                    return None, e, None
    return None, last_err, None

In [ ]:
import re
from pathlib import Path

def sub(src, old, new):
    assert old in src, f"pattern not found: {old[:60]!r}"
    return src.replace(old, new, 1)

# ---------- config.py: backup models ----------
p = Path("config.py"); src = p.read_text(encoding="utf-8")
if "GENERATOR_FALLBACKS" not in src:
    m = re.search(r"(?m)^GENERATOR_MODEL\s*=.*$", src)
    assert m, "GENERATOR_MODEL line not found in config.py"
    extra = (
        "\n# Backup models used when the main one is overloaded (comma separated in the env var).\n"
        "# By default each role falls back to the other role's model. Use names your key can access.\n"
        'GENERATOR_FALLBACKS = [x.strip() for x in os.getenv("GENERATOR_FALLBACKS", ROUTER_MODEL).split(",") if x.strip()]\n'
        'ROUTER_FALLBACKS = [x.strip() for x in os.getenv("ROUTER_FALLBACKS", GENERATOR_MODEL).split(",") if x.strip()]'
    )
    p.write_text(src[:m.end()] + extra + src[m.end():], encoding="utf-8")
print("config.py patched")

# ---------- router.py ----------
p = Path("router.py"); src = p.read_text(encoding="utf-8")
head = src.split("# ---------- public API ----------")[0]
head = sub(head, "import config\n", "import config\nimport llm_utils\n")
head = sub(head, "def _call_router(client, system: str, contents: str) -> str:",
                 "def _call_router(client, system: str, contents: str, model: str = None) -> str:")
head = sub(head, "model=config.ROUTER_MODEL,", "model=model or config.ROUTER_MODEL,")

tail = r'''# ---------- public API ----------

def route(question: str, history, loaded, client, last_ids=None) -> Route:
    """loaded = loader.LoadResult. Never raises.
    Quota / overload errors -> retries, then a fallback model, then intent="error" (never a silent "general").
    Other failures (malformed JSON...) -> intent="general"."""
    system = SYSTEM_TEMPLATE.format(
        years=", ".join(map(str, loaded.years)),
        catalog=format_catalog(loaded.catalog),
    )
    contents = (
        f"Conversation so far:\n{format_history(history)}\n\n"
        f"Last exercise(s) discussed: {', '.join(last_ids) if last_ids else '(none)'}\n\n"
        f"New student message:\n{question}"
    )
    by_id = loaded.by_id()
    models = list(dict.fromkeys([config.ROUTER_MODEL] + list(getattr(config, "ROUTER_FALLBACKS", []))))

    last_err = None
    for _ in range(2):   # the second pass only matters for malformed JSON
        raw, err, used = llm_utils.call_with_retry(lambda m: _call_router(client, system, contents, m), models)
        if err is not None:
            last_err = err
            break
        try:
            result = _validate(RouterDecision.model_validate_json(raw), by_id)
            if used != models[0]:
                result.notes.append(f"used fallback model: {used}")
            return result
        except Exception as e:
            last_err = e

    if llm_utils.is_quota_error(last_err) or llm_utils.is_transient_error(last_err):
        return Route(
            intent="error",
            standalone_question=question,
            clarification="الخدمة مشغولة حاليا. عاود المحاولة بعد دقيقة.",
            notes=[f"service busy: {last_err!r}"[:300]],
        )
    return Route(intent="general", standalone_question=question,
                 notes=[f"router failed, fallback to general: {last_err!r}"[:300]])
'''
p.write_text(head + tail, encoding="utf-8")
print("router.py patched")

# ---------- generator.py ----------
p = Path("generator.py"); src = p.read_text(encoding="utf-8")
old_import = re.search(r"from router import \(.*?\)\n", src, re.S)
assert old_import, "router import not found in generator.py"
src = src.replace(old_import.group(0), "import llm_utils\nfrom router import format_history\n", 1)
src = sub(src, "def _call_gemini(client, system: str, parts: list) -> str:",
               "def _call_gemini(client, system: str, parts: list, model: str = None) -> str:")
src = sub(src, "model=config.GENERATOR_MODEL,", "model=model or config.GENERATOR_MODEL,")

marker = "    # 3) call, with patience for quota errors"
assert marker in src, "step 3 marker not found in generator.py"
new_block = '''    # 3) call Gemini: retries, back-off and fallback model are handled by llm_utils
    models = list(dict.fromkeys([config.GENERATOR_MODEL] + list(getattr(config, "GENERATOR_FALLBACKS", []))))
    out, err, used = llm_utils.call_with_retry(lambda m: _call_gemini(client, system, parts, m), models)

    if err is None:
        if used != models[0]:
            notes.append(f"used fallback model: {used}")
        if out and out.strip():
            return Answer(text=out, ids=route.ids if chunks else [], figures=figures,
                          used_gemini=True, notes=notes)
        return Answer(text="ما قدرتش نولّد جواب لهذا السؤال. جرّب تعيد صياغته.",
                      notes=notes + ["empty response from Gemini"])

    busy = llm_utils.is_quota_error(err) or llm_utils.is_transient_error(err)
    msg = ("الخدمة مشغولة حاليا. عاود المحاولة بعد دقيقة." if busy
           else "صرات مشكلة تقنية. عاود المحاولة من فضلك.")
    return Answer(text=msg, notes=notes + [f"generation failed: {err!r}"[:300]])
'''
p.write_text(src.split(marker)[0] + new_block, encoding="utf-8")
print("generator.py patched")

In [ ]:
import importlib, time, config, llm_utils, router, generator
for m in (config, llm_utils, router, generator):
    importlib.reload(m)
print("models:", config.ROUTER_MODEL, config.ROUTER_FALLBACKS, "|", config.GENERATOR_MODEL, config.GENERATOR_FALLBACKS)

ask("حلي لي التمرين الأول تاع باك 2022 الموضوع الأول", mode="hints")

In [ ]:
import re, importlib
from pathlib import Path
import generator

p = Path("generator.py"); src = p.read_text(encoding="utf-8")
new = '''MODE_HINTS = """
MODE: HINTS ONLY (progressive)
- Do NOT give the full solution. Do NOT state final results, nor numeric values read from the figure or the correction.
- If the student asks about the whole exercise: give a very short overview (which lessons are needed),
  then hints for the FIRST sub-question only. Invite the student to try it and to ask for the next one.
- If the student asks about one sub-question: give hints for that sub-question only.
- Each hint: (1) the lesson or theorem to use, (2) the first step or a guiding question, (3) one short nudge.
- End by saying the student can switch to the "full solution" option (the mode selector) to see the complete answer.
"""'''
pattern = re.compile(r'MODE_HINTS = """.*?"""', re.S)
assert pattern.search(src), "MODE_HINTS block not found in generator.py"
p.write_text(pattern.sub(lambda m: new, src, count=1), encoding="utf-8")
importlib.reload(generator)
print("MODE_HINTS updated")

In [ ]:
import time
time.sleep(15)
r, a = ask("حلي لي التمرين الأول تاع باك 2022 الموضوع الأول", mode="full")

# نعرض التصحيح الرسمي للمقارنة
print("\n" + "="*60 + "\nOFFICIAL CORRECTION (من الـ dataset):\n" + "="*60)
print(loaded.by_id()["2022_s1_ex1"]["solution"])

In [ ]:
import importlib
from pathlib import Path
import generator

p = Path("generator.py"); src = p.read_text(encoding="utf-8")
old = '        figures = [f["path"] for f in chunks[0]["figures_sujet"] if Path(f["path"]).exists()]\n'
new = ('        shown = list(chunks[0]["figures_sujet"]) + (list(chunks[0]["figures_corr"]) if mode == "full" else [])\n'
       '        figures = [f["path"] for f in shown if Path(f["path"]).exists()]   # corr. figures only in full mode\n')
assert old in src, "figures line not found in generator.py"
p.write_text(src.replace(old, new, 1), encoding="utf-8")
importlib.reload(generator)
print("figures patch applied")

In [ ]:
%%writefile assistant.py
"""Glue between loader, router and generator. No UI code here, so any front-end
(Gradio, Telegram, an API...) can reuse it."""
import config
import generator
import loader
import router


class Assistant:
    def __init__(self, client=None, loaded=None):
        self.loaded = loaded or loader.load_all()
        if client is None:
            from google import genai
            client = genai.Client(api_key=config.get_api_key())
        self.client = client

    def reload_data(self):
        """Re-scan DATA_DIR (cheap when nothing changed thanks to the cache fingerprint)."""
        self.loaded = loader.load_all()
        return self.loaded

    def ask(self, question, history, mode="full", last_ids=None):
        """Returns (route, answer). Never raises."""
        r = router.route(question, history, self.loaded, self.client, last_ids=last_ids)
        a = generator.generate(r, self.loaded, self.client, mode=mode, history=history)
        return r, a

    def full_solution(self, route, history):
        """Same request as before but in full mode, reusing the router's decision (1 API call)."""
        return generator.generate(route, self.loaded, self.client, mode="full", history=history)

In [ ]:
%%writefile app.py
"""Gradio UI for the BAC assistant.

    python app.py                      # local
    GRADIO_SHARE=1 python app.py       # public link (Kaggle / Colab)
    import app; app.launch(share=True) # from a notebook
"""
import inspect
import os

import gradio as gr

import config
from assistant import Assistant

MODE_CHOICES = [("حل كامل", "full"), ("تلميحات فقط", "hints")]
LATEX = [
    {"left": "$$", "right": "$$", "display": True},
    {"left": "\\[", "right": "\\]", "display": True},
    {"left": "\\(", "right": "\\)", "display": False},
    {"left": "$", "right": "$", "display": False},
]
EXAMPLES = [
    "حلي لي التمرين الأول تاع باك 2022 الموضوع الأول",
    "عطيني تمارين على الأعداد المركبة",
    "شرحلي كيفاش نحسب نهاية دالة أسية",
    "واش هو مشتق x^2 * e^x ؟",
]
EXPANDABLE = ("solve_exercise", "solve_subquestion")


# ---------- helpers ----------

def _kw(cls, **kwargs):
    """Pass only the arguments this Gradio version supports (the API changed between 4/5/6)."""
    allowed = inspect.signature(cls.__init__).parameters
    return {k: v for k, v in kwargs.items() if k in allowed}


def _skip():
    return gr.skip() if hasattr(gr, "skip") else gr.update()


def _text_of(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return str(content.get("text", ""))
    if isinstance(content, (list, tuple)):
        return " ".join(_text_of(c) for c in content)
    return ""


def clean_history(chat):
    """Gradio chat history -> plain [{'role','content'}] with text only."""
    out = []
    for m in chat or []:
        text = _text_of(m.get("content")).strip()
        if text and m.get("role") in ("user", "assistant"):
            out.append({"role": m["role"], "content": text})
    return out


def status_text(assistant) -> str:
    L = assistant.loaded
    s = f"📚 قاعدة البيانات: **{len(L.years)} سنوات** ({', '.join(map(str, L.years))}) • **{len(L.chunks)} تمرين**"
    if L.warnings:
        s += f"\n\n⚠️ {len(L.warnings)} تنبيه في البيانات:\n" + "\n".join(f"- {w}" for w in L.warnings[:10])
    return s


def _gallery(paths):
    return gr.Gallery(value=paths or None, visible=bool(paths))


# ---------- handlers ----------

def make_handlers(assistant):
    def respond(message, chat, mode, last_ids, last_route):
        message = (message or "").strip()
        if not message:
            return "", _skip(), _skip(), _skip(), _skip(), _skip()
        chat = list(chat or [])
        mode = mode if mode in ("full", "hints") else "full"

        route, ans = assistant.ask(message, clean_history(chat), mode, last_ids)

        chat += [{"role": "user", "content": message}, {"role": "assistant", "content": ans.text}]
        if route.ids and route.intent in EXPANDABLE + ("explain_concept",):
            last_ids = list(route.ids)          # remembered for follow-ups like "والسؤال 2؟"
        can_expand = mode == "hints" and ans.used_gemini and route.intent in EXPANDABLE
        return "", chat, _gallery(ans.figures), last_ids, (route if can_expand else None), gr.Button(visible=can_expand)

    def full_solution(chat, last_route):
        if last_route is None:
            return _skip(), _skip(), gr.Button(visible=False), None
        chat = list(chat or [])
        ans = assistant.full_solution(last_route, clean_history(chat))
        chat += [{"role": "user", "content": "اعرض الحل الكامل"}, {"role": "assistant", "content": ans.text}]
        return chat, _gallery(ans.figures), gr.Button(visible=False), None

    def refresh():
        try:
            assistant.reload_data()
            return status_text(assistant)
        except Exception as e:
            return f"❌ تعذر تحميل البيانات: {e}"

    return respond, full_solution, refresh


# ---------- layout ----------

def build_app(assistant):
    respond, full_solution, refresh = make_handlers(assistant)

    with gr.Blocks(title="مساعد الباك – رياضيات") as demo:
        gr.Markdown("# 🎓 مساعد الباك في الرياضيات\nاسألني على تمارين الباك، أو أي سؤال في الرياضيات.")
        status = gr.Markdown(status_text(assistant))

        with gr.Row():
            with gr.Column(scale=3):
                chatbot = gr.Chatbot(**_kw(gr.Chatbot, label="المحادثة", height=520, rtl=True,
                                           latex_delimiters=LATEX, show_copy_button=True, type="messages"))
                with gr.Row():
                    msg = gr.Textbox(**_kw(gr.Textbox, placeholder="اكتب سؤالك هنا…", show_label=False,
                                           rtl=True, scale=6, lines=1))
                    send = gr.Button("إرسال", variant="primary", scale=1)
                full_btn = gr.Button("📖 اعرض الحل الكامل", visible=False)
            with gr.Column(scale=1):
                mode = gr.Radio(choices=MODE_CHOICES, value="full", label="طريقة الإجابة على التمارين")
                gallery = gr.Gallery(**_kw(gr.Gallery, label="أشكال التمرين", visible=False, columns=1))
                refresh_btn = gr.Button("🔄 تحديث قاعدة البيانات")

        gr.Examples(examples=EXAMPLES, inputs=msg, label="أمثلة")

        last_ids, last_route = gr.State([]), gr.State(None)
        inputs = [msg, chatbot, mode, last_ids, last_route]
        outputs = [msg, chatbot, gallery, last_ids, last_route, full_btn]
        msg.submit(respond, inputs, outputs)
        send.click(respond, inputs, outputs)
        full_btn.click(full_solution, [chatbot, last_route], [chatbot, gallery, full_btn, last_route])
        refresh_btn.click(refresh, None, status)
    return demo


def launch(share=None, **kwargs):
    assistant = Assistant()
    demo = build_app(assistant)
    if share is None:
        share = bool(os.getenv("GRADIO_SHARE"))
    demo.queue().launch(share=share, allowed_paths=[str(config.DATA_DIR.resolve())], **kwargs)
    return demo


if __name__ == "__main__":
    launch()

In [ ]:
import importlib, config, llm_utils, loader, router, generator, assistant, app
for m in (config, llm_utils, loader, router, generator, assistant, app):
    importlib.reload(m)

demo = app.launch(share=True, debug=False)

In [ ]:
import time, functools, config, router, generator

events = []
def timed(name, fn):
    @functools.wraps(fn)
    def w(*a, **k):
        model = k.get("model") or (a[3] if len(a) > 3 else None)
        t = time.time()
        try:
            out = fn(*a, **k)
            events.append((name, model, round(time.time() - t, 2), "ok"))
            return out
        except Exception as e:
            events.append((name, model, round(time.time() - t, 2), f"{type(e).__name__} {getattr(e, 'code', '')}"))
            raise
    return w

_orig = (router._call_router, generator._call_gemini)
router._call_router = timed("router", router._call_router)
generator._call_gemini = timed("generator", generator._call_gemini)

q = "حلي لي التمرين الأول تاع باك 2022 الموضوع الأول"
t0 = time.time()
r = router.route(q, [], loaded, client)
t1 = time.time()
a = generator.generate(r, loaded, client, mode="full")
t2 = time.time()
router._call_router, generator._call_gemini = _orig        # نرجعو الدوال الأصلية

print(f"router    : {t1 - t0:.1f}s")
print(f"generator : {t2 - t1:.1f}s")
print(f"TOTAL     : {t2 - t0:.1f}s")
print("\nكل محاولة:")
for ev in events:
    print("  ", ev)
busy = sum(e[2] for e in events)
print(f"\nوقت الاستدعاءات الفعلي: {busy:.1f}s | وقت الانتظار بين المحاولات: {max((t2 - t0) - busy, 0):.1f}s")
print("notes:", r.notes + a.notes)
print("طول الجواب:", len(a.text), "حرف")

In [ ]:
for name in (config.ROUTER_MODEL, config.GENERATOR_MODEL):
    t = time.time()
    try:
        resp = client.models.generate_content(model=name, contents="2+2 ?")
        print(f"{name}: {time.time() - t:.1f}s")
        print("   ", resp.usage_metadata)
    except Exception as e:
        print(f"{name}: ERROR {type(e).__name__} {str(e)[:120]}")